In [1]:
import os
os.getcwd()

'/home/rohit13595/rl-projects/pro3-multifeedbackRLHF/RL-withMultiHumanFeedback/scripts_hopper'

In [ ]:
import gymnasium as gym
import numpy as np
import time
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy

# --- Model Configuration ---
# 1. DIRECT PATH TO THE MODEL ZIP FILE (YOU MUST VERIFY THIS PATH)
# This path is based on your folder structure: agents/ALGO/EXP_NAME/best_model.zip
MODEL_ZIP_PATH = "./agents/RL_sac_Hopper-v5_1789_evaluative_1789/sac/Hopper-v5_1789/best_model.zip"

# 2. Environment and Evaluation Params
ENV_ID = "Hopper-v5"
EVAL_EPISODES = 1000
SEED = 1789


def evaluate_and_run_policy(model_path: str, env_id: str):
    """Loads a model, runs it in the environment, and displays the result."""
    
    print(f"--- Attempting to load model from: {model_path} ---")
    
    # 1. Create the Environment (render_mode="human" attempts to show the video)
    try:
        env = gym.make(env_id, render_mode="human")
    except Exception as e:
        print(f"Error creating environment: {e}")
        return

    # 2. Load the Policy (SAC Model)
    try:
        # Load the policy. The architecture is retrieved from the ZIP file itself.
        # We pass env=env to automatically configure the model for the environment.
        model = SAC.load(model_path, env=env, device="auto")
    except Exception as e:
        print(f"Error loading model. Check model path and file integrity.")
        print(f"Details: {e}")
        env.close()
        return

    print(f"Policy loaded successfully. Starting evaluation over {EVAL_EPISODES} episodes.")
    
    # 3. Evaluate Performance (Mean Reward)
    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=EVAL_EPISODES,
        deterministic=True, # Use deterministic actions for a stable performance measure
    )
    
    print("\n========================================")
    print(f"FINAL EVALUATION RESULT ({EVAL_EPISODES} episodes):")
    print(f"Mean Reward (GT): {mean_reward:.2f} ± {std_reward:.2f}")
    print("========================================")

    # 4. Run loop for live visualization (video)
    print("\nStarting live run...")
    
    observation, info = env.reset(seed=SEED)
    episode_reward = 0.0
    
    while True:
        # Predict the action using the deterministic policy
        action, _states = model.predict(observation, deterministic=True)
        observation, reward, terminated, truncated, info = env.step(action)
        
        episode_reward += reward
        env.render()
        # time.sleep(0.01)

        if terminated or truncated:
            print(f"Episode finished. Final Reward: {episode_reward:.2f}")
            observation, info = env.reset(seed=SEED)
            episode_reward = 0.0
            
            # Stop after the first episode for simplicity
            break
            
    env.close()


if __name__ == "__main__":
    evaluate_and_run_policy(MODEL_ZIP_PATH, ENV_ID)